# Debug: residual short-horizon FINE magnitude −log p(m)

**Guide:** [`docs/RESIDUAL_SHORT_MAG_LL_DEBUG.md`](../docs/RESIDUAL_SHORT_MAG_LL_DEBUG.md) (Phase A).

After the FeatureState fix, slides use `_short_v2` for 0.5 / 1 / 4 d and the long tree for 7 / 60 / 300 d. FINE still loses to ETAS on **forecast** magnitude LL at short horizons while beating GR on **observed test** events.

This notebook freezes residual facts from existing `magnitude_likelihoods.csv` caches (no re-run):

1. Aggregation parity with the slide plot (per-seed mean vs pooled)
2. Per-seed outlier / missing-seed audit
3. Heavy-tail rates vs horizon
4. First vs later events within a window
5. Same drawn magnitude: MAGNET density vs GR

**Exit question:** is the residual mainly (i) heavy FINE draws, (ii) a few bad seeds, (iii) later-in-window events, or (iv) plot aggregation?


In [ ]:
from __future__ import annotations

%matplotlib inline

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "etas").is_dir():
    REPO_ROOT = Path("/a/home/cc/students/csguests/neriberman/Repos/etas")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import etas.rolling_analysis as rolling_analysis

METHODS = ("etas", "FINE")
DENSITY_COL = {"etas": "etas_likelihood", "FINE": "fine_likelihood"}
COLORS = {"etas": "#284473", "FINE": "#E6225B"}
LABELS = {"etas": "ETAS", "FINE": "FINE"}

EXPERIMENTS = [
    {
        "output_root": REPO_ROOT / "outputs/rolling_continuation_hauksson_pre_ridgecrest_short_v2",
        "horizon_days": [0.5, 1.0, 4.0],
        "label": "short_v2",
    },
    {
        "output_root": REPO_ROOT / "outputs/rolling_continuation_hauksson_pre_ridgecrest",
        "horizon_days": [7.0, 60.0, 300.0],
        "label": "long",
    },
]

EPS = 1e-300


In [ ]:
def neg_log(density: np.ndarray | pd.Series) -> np.ndarray:
    values = np.asarray(density, dtype=float)
    return -np.log(np.maximum(values, EPS))


def mean_neg_log(density: np.ndarray | pd.Series) -> float:
    values = neg_log(density)
    values = values[np.isfinite(values)]
    return float(values.mean()) if values.size else float("nan")


def load_specs(experiments=EXPERIMENTS):
    specs = []
    for experiment in experiments:
        root = experiment["output_root"]
        if not root.is_dir():
            continue
        for days in experiment["horizon_days"]:
            horizon_dir = rolling_analysis.horizon_directory(root, float(days))
            csv_path = horizon_dir / "analysis_cache" / "magnitude_likelihoods.csv"
            if not csv_path.is_file():
                continue
            specs.append({
                "horizon_days": float(days),
                "output_root": root,
                "horizon_dir": horizon_dir,
                "csv_path": csv_path,
                "tree": experiment["label"],
            })
    return sorted(specs, key=lambda item: item["horizon_days"])


def load_likelihoods(spec: dict) -> pd.DataFrame:
    frame = pd.read_csv(spec["csv_path"])
    frame["horizon_days"] = float(spec["horizon_days"])
    frame["tree"] = spec["tree"]
    return frame


def expected_seeds(spec: dict) -> set[int]:
    """Seeds advertised by step_000 files keys (FINE/k and etas/k)."""
    step_path = spec["horizon_dir"] / "analysis_cache" / "steps" / "step_000.json"
    if not step_path.is_file():
        step_path = next((spec["horizon_dir"] / "analysis_cache" / "steps").glob("step_*.json"), None)
    if step_path is None or not Path(step_path).is_file():
        return set()
    meta = json.loads(Path(step_path).read_text())
    seeds = set()
    for key in meta.get("files", {}):
        if "/" not in key:
            continue
        _method, seed_text = key.split("/", 1)
        try:
            seeds.add(int(seed_text))
        except ValueError:
            continue
    return seeds


def load_window_bounds(spec: dict) -> pd.DataFrame:
    steps_dir = spec["horizon_dir"] / "analysis_cache" / "steps"
    rows = []
    for path in sorted(steps_dir.glob("step_*.json")):
        meta = json.loads(path.read_text())
        start = pd.Timestamp(meta["forecast_start"])
        end = pd.Timestamp(meta["forecast_end"])
        rows.append({
            "step_index": int(meta["step_index"]),
            "start_unix": int(start.timestamp()),
            "end_unix": int(end.timestamp()),
        })
    return pd.DataFrame(rows).sort_values("step_index").reset_index(drop=True)


def assign_windows(forecast: pd.DataFrame, windows: pd.DataFrame) -> pd.DataFrame:
    """Map each forecast event to a rolling window by time_unix ∈ [start, end)."""
    if forecast.empty or windows.empty:
        out = forecast.copy()
        out["step_index"] = np.nan
        out["event_index_in_window"] = np.nan
        return out
    times = forecast["time_unix"].to_numpy(dtype=np.int64)
    starts = windows["start_unix"].to_numpy(dtype=np.int64)
    ends = windows["end_unix"].to_numpy(dtype=np.int64)
    # searchsorted on starts: event belongs to last window with start <= t
    idx = np.searchsorted(starts, times, side="right") - 1
    valid = (idx >= 0) & (idx < len(starts)) & (times < ends[np.clip(idx, 0, len(ends) - 1)])
    step = np.full(times.shape, -1, dtype=np.int64)
    step[valid] = windows["step_index"].to_numpy(dtype=np.int64)[idx[valid]]
    out = forecast.copy()
    out["step_index"] = step
    out = out.loc[out["step_index"] >= 0].copy()
    out = out.sort_values(["source", "seed", "step_index", "time_unix"])
    out["event_index_in_window"] = out.groupby(["source", "seed", "step_index"]).cumcount() + 1
    return out


SPECS = load_specs()
display(Markdown(
    "**Horizons loaded:** "
    + ", ".join(f"{s['horizon_days']:g} d ({s['tree']})" for s in SPECS)
    + f"  \n**Repo:** `{REPO_ROOT}`"
))
assert SPECS, "No magnitude_likelihoods.csv found under EXPERIMENTS roots"


## A1 — Aggregation parity (slide vs pooled)

The slide notebook uses **mean of per-seed mean log-densities**, then negates. A pooled event mean can disagree when seed catalog sizes differ a lot.


In [ ]:
agg_rows = []
frames_by_horizon: dict[float, pd.DataFrame] = {}

for spec in SPECS:
    frame = load_likelihoods(spec)
    frames_by_horizon[spec["horizon_days"]] = frame
    observed = frame.loc[(frame["population"] == "test") & (frame["source"] == "observed")]
    forecasts = frame.loc[frame["population"] == "forecast"]
    for method in METHODS:
        col = DENSITY_COL[method]
        method_fc = forecasts.loc[forecasts["source"] == method]
        seed_means = method_fc.groupby("seed")[col].apply(mean_neg_log)
        seed_means = seed_means[np.isfinite(seed_means)]
        pooled = mean_neg_log(method_fc[col]) if len(method_fc) else float("nan")
        test_nll = mean_neg_log(observed[col]) if len(observed) else float("nan")
        sizes = method_fc.groupby("seed").size()
        agg_rows.append({
            "horizon_days": spec["horizon_days"],
            "tree": spec["tree"],
            "method": method,
            "n_seeds": int(seed_means.size),
            "n_events": int(len(method_fc)),
            "events_per_seed_median": float(sizes.median()) if len(sizes) else float("nan"),
            "events_per_seed_min": int(sizes.min()) if len(sizes) else 0,
            "events_per_seed_max": int(sizes.max()) if len(sizes) else 0,
            "slide_mean_neg_log": float(seed_means.mean()) if seed_means.size else float("nan"),
            "slide_std_neg_log": float(seed_means.std(ddof=1)) if seed_means.size > 1 else 0.0,
            "pooled_mean_neg_log": pooled,
            "pooled_minus_slide": pooled - float(seed_means.mean()) if seed_means.size else float("nan"),
            "test_mean_neg_log": test_nll,
        })

agg = pd.DataFrame(agg_rows)
display(agg.round(4))

fig, ax = plt.subplots(figsize=(8, 4.2))
for method in METHODS:
    sub = agg.loc[agg["method"] == method].sort_values("horizon_days")
    ax.plot(
        sub["horizon_days"], sub["slide_mean_neg_log"],
        marker="o" if method == "etas" else "s",
        color=COLORS[method], label=f"{LABELS[method]} slide (mean of seeds)",
    )
    ax.plot(
        sub["horizon_days"], sub["pooled_mean_neg_log"],
        marker="o" if method == "etas" else "s",
        color=COLORS[method], linestyle="--", alpha=0.7,
        label=f"{LABELS[method]} pooled events",
    )
    ax.axhline(sub["test_mean_neg_log"].iloc[0], color=COLORS[method], linestyle=":", alpha=0.5)
ax.set_xscale("log")
ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel(r"Mean $-\log p(m)$ (lower better)")
ax.set_title("Aggregation parity: slide vs pooled")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

display(Markdown(
    "Dotted horizontal lines are **observed test** −log p(m) (same on every tree). "
    "If slide and pooled disagree a lot, seed catalog sizes are skewed."
))


## A2 — Per-seed outlier / missing-seed audit


In [ ]:
seed_rows = []
missing_rows = []

for spec in SPECS:
    frame = frames_by_horizon[spec["horizon_days"]]
    forecasts = frame.loc[frame["population"] == "forecast"]
    advertised = expected_seeds(spec)
    for method in METHODS:
        col = DENSITY_COL[method]
        method_fc = forecasts.loc[forecasts["source"] == method]
        present = set(int(s) for s in method_fc["seed"].unique())
        empty_npz = []
        events_dir = spec["horizon_dir"] / "analysis_cache" / "events" / method
        if events_dir.is_dir():
            for path in events_dir.glob("seed_*.npz"):
                seed = int(path.stem.split("_")[1])
                n = int(np.load(path)["magnitude"].shape[0])
                if n == 0:
                    empty_npz.append(seed)
        missing = sorted(advertised - present) if advertised else sorted(set(empty_npz) - present)
        missing_rows.append({
            "horizon_days": spec["horizon_days"],
            "method": method,
            "advertised_n": len(advertised) if advertised else None,
            "present_n": len(present),
            "missing_seeds": missing,
            "empty_npz_seeds": sorted(empty_npz),
        })
        for seed, group in method_fc.groupby("seed"):
            seed_rows.append({
                "horizon_days": spec["horizon_days"],
                "method": method,
                "seed": int(seed),
                "n_events": int(len(group)),
                "mean_neg_log": mean_neg_log(group[col]),
                "median_mag": float(group["magnitude"].median()),
                "frac_m_ge_5": float((group["magnitude"] >= 5.0).mean()),
            })

seed_stats = pd.DataFrame(seed_rows)
missing = pd.DataFrame(missing_rows)
display(Markdown("### Missing / empty seeds"))
display(missing)

focus_horizons = [0.5, 7.0]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=False)
for ax, days in zip(axes, focus_horizons):
    sub = seed_stats.loc[np.isclose(seed_stats["horizon_days"], days)]
    if sub.empty:
        ax.set_title(f"{days:g} d (missing)")
        continue
    for method in METHODS:
        m = sub.loc[sub["method"] == method]
        ax.hist(
            m["mean_neg_log"], bins=12, alpha=0.55,
            color=COLORS[method], label=f"{LABELS[method]} (n={len(m)})",
        )
    ax.set_title(rf"Seed-mean $-\log p(m)$ @ {days:g} d")
    ax.set_xlabel(r"Seed mean $-\log p(m)$")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("Number of seeds")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, days in zip(axes, focus_horizons):
    sub = seed_stats.loc[np.isclose(seed_stats["horizon_days"], days) & (seed_stats["method"] == "FINE")]
    if sub.empty:
        continue
    ax.scatter(sub["n_events"], sub["mean_neg_log"], c=COLORS["FINE"], s=40)
    for _, row in sub.iterrows():
        if row["n_events"] < 50 or row["mean_neg_log"] > 1.0 or row["n_events"] > 5000:
            ax.annotate(str(int(row["seed"])), (row["n_events"], row["mean_neg_log"]), fontsize=8)
    ax.set_xscale("log")
    ax.set_title(f"FINE seed size vs −log p @ {days:g} d")
    ax.set_xlabel("Events per seed")
    ax.set_ylabel(r"Seed mean $-\log p(m)$")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

display(Markdown("### Worst FINE seeds by horizon (top 5 mean −log p)"))
for days in [s["horizon_days"] for s in SPECS]:
    sub = seed_stats.loc[
        np.isclose(seed_stats["horizon_days"], days) & (seed_stats["method"] == "FINE")
    ].sort_values("mean_neg_log", ascending=False).head(5)
    display(Markdown(f"**{days:g} d**"))
    display(sub.round(4))


## A3 — Heavy-tail audit


In [ ]:
tail_rows = []
for spec in SPECS:
    frame = frames_by_horizon[spec["horizon_days"]]
    forecasts = frame.loc[frame["population"] == "forecast"]
    observed = frame.loc[(frame["population"] == "test") & (frame["source"] == "observed")]
    for method, source_frame in [
        ("observed_test", observed),
        ("etas", forecasts.loc[forecasts["source"] == "etas"]),
        ("FINE", forecasts.loc[forecasts["source"] == "FINE"]),
    ]:
        if source_frame.empty:
            continue
        mag = source_frame["magnitude"].to_numpy(dtype=float)
        tail_rows.append({
            "horizon_days": spec["horizon_days"],
            "method": method,
            "n_events": int(mag.size),
            "median_mag": float(np.median(mag)),
            "p90_mag": float(np.quantile(mag, 0.90)),
            "p99_mag": float(np.quantile(mag, 0.99)),
            "frac_m_ge_4": float((mag >= 4.0).mean()),
            "frac_m_ge_5": float((mag >= 5.0).mean()),
        })

tails = pd.DataFrame(tail_rows)
display(tails.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, col, ylabel in zip(
    axes,
    ["frac_m_ge_5", "p90_mag"],
    [r"Fraction $M \geq 5$", "p90 magnitude"],
):
    for method in ("observed_test", "etas", "FINE"):
        sub = tails.loc[tails["method"] == method].sort_values("horizon_days")
        style = {
            "observed_test": dict(color="#111111", marker="^", linestyle=":"),
            "etas": dict(color=COLORS["etas"], marker="o"),
            "FINE": dict(color=COLORS["FINE"], marker="s"),
        }[method]
        ax.plot(sub["horizon_days"], sub[col], label=method, **style)
    ax.set_xscale("log")
    ax.set_xlabel("Forecast horizon (days)")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
axes[0].set_title("Heavy-tail rate vs horizon")
axes[1].set_title("Upper magnitude quantile vs horizon")
plt.tight_layout()
plt.show()


## A4 — Position in window (first vs later events)

Assign each forecast event to a rolling window via `analysis_cache/steps/step_*.json` bounds, then compare −log p(m) for event index 1 vs later indices. If first events look healthy but later ones do not, synthetic-history drift inside the window is implicated.


In [ ]:
# Window assignment is heavier for short horizons (many steps). Cache per horizon.
position_rows = []
first_vs_later = []

for spec in SPECS:
    frame = frames_by_horizon[spec["horizon_days"]]
    forecasts = frame.loc[frame["population"] == "forecast"].copy()
    windows = load_window_bounds(spec)
    assigned = assign_windows(forecasts, windows)
    n_unmapped = len(forecasts) - len(assigned)
    for method in METHODS:
        col = DENSITY_COL[method]
        sub = assigned.loc[assigned["source"] == method]
        if sub.empty:
            continue
        first = sub.loc[sub["event_index_in_window"] == 1]
        later = sub.loc[sub["event_index_in_window"] > 1]
        first_vs_later.append({
            "horizon_days": spec["horizon_days"],
            "method": method,
            "n_windows_with_event": int(first["step_index"].nunique()),
            "n_first": int(len(first)),
            "n_later": int(len(later)),
            "n_unmapped_events": int(n_unmapped),
            "first_mean_neg_log": mean_neg_log(first[col]) if len(first) else float("nan"),
            "later_mean_neg_log": mean_neg_log(later[col]) if len(later) else float("nan"),
            "first_frac_m_ge_5": float((first["magnitude"] >= 5).mean()) if len(first) else float("nan"),
            "later_frac_m_ge_5": float((later["magnitude"] >= 5).mean()) if len(later) else float("nan"),
        })
        # binned by event index (cap for readability)
        capped = sub.copy()
        capped["idx_bin"] = np.minimum(capped["event_index_in_window"], 10)
        for idx, group in capped.groupby("idx_bin"):
            position_rows.append({
                "horizon_days": spec["horizon_days"],
                "method": method,
                "event_index_bin": int(idx),
                "n": int(len(group)),
                "mean_neg_log": mean_neg_log(group[col]),
            })

pos_summary = pd.DataFrame(first_vs_later)
pos_bins = pd.DataFrame(position_rows)
display(Markdown("### First vs later events"))
display(pos_summary.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, days in zip(axes, [0.5, 7.0]):
    sub = pos_bins.loc[np.isclose(pos_bins["horizon_days"], days)]
    for method in METHODS:
        m = sub.loc[sub["method"] == method].sort_values("event_index_bin")
        if m.empty:
            continue
        ax.plot(
            m["event_index_bin"], m["mean_neg_log"],
            marker="o" if method == "etas" else "s",
            color=COLORS[method], label=LABELS[method],
        )
    ax.set_xlabel("Event index in window (10 = 10+)")
    ax.set_ylabel(r"Mean $-\log p(m)$")
    ax.set_title(f"By position in window @ {days:g} d")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## A5 — Same drawn magnitude: MAGNET vs GR on FINE forecasts

For each FINE synthetic event the cache stores both `fine_likelihood` and `etas_likelihood`. If MAGNET density is *better* than GR on the same drawn `m` but the slide still loses, the problem is the **draws** (too heavy / wrong locations in feature space), not MAGNET failing to score those magnitudes.


In [ ]:
same_m_rows = []
for spec in SPECS:
    frame = frames_by_horizon[spec["horizon_days"]]
    fine = frame.loc[(frame["population"] == "forecast") & (frame["source"] == "FINE")].copy()
    if fine.empty:
        continue
    fine["nll_fine"] = neg_log(fine["fine_likelihood"])
    fine["nll_etas"] = neg_log(fine["etas_likelihood"])
    fine["magnet_wins"] = fine["nll_fine"] < fine["nll_etas"]
    same_m_rows.append({
        "horizon_days": spec["horizon_days"],
        "n_events": int(len(fine)),
        "mean_nll_magnet": float(fine["nll_fine"].mean()),
        "mean_nll_gr_on_same_m": float(fine["nll_etas"].mean()),
        "frac_magnet_wins": float(fine["magnet_wins"].mean()),
        "mean_nll_magnet_on_m_ge_5": float(fine.loc[fine["magnitude"] >= 5, "nll_fine"].mean())
        if (fine["magnitude"] >= 5).any() else float("nan"),
        "mean_nll_gr_on_m_ge_5": float(fine.loc[fine["magnitude"] >= 5, "nll_etas"].mean())
        if (fine["magnitude"] >= 5).any() else float("nan"),
        "frac_m_ge_5": float((fine["magnitude"] >= 5).mean()),
    })

same_m = pd.DataFrame(same_m_rows)
display(same_m.round(4))

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(same_m["horizon_days"], same_m["mean_nll_magnet"], "s-", color=COLORS["FINE"], label="MAGNET on FINE draws")
ax.plot(same_m["horizon_days"], same_m["mean_nll_gr_on_same_m"], "o-", color=COLORS["etas"], label="GR on same FINE draws")
ax.set_xscale("log")
ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel(r"Mean $-\log p(m)$")
ax.set_title("Same FINE magnitudes scored by MAGNET vs GR")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Phase A exit summary

Fill this after running all cells. Point to the strongest evidence for one primary cause.


In [ ]:
summary_bits = []

# Aggregation gap at 0.5 d
row_05 = agg.loc[np.isclose(agg["horizon_days"], 0.5) & (agg["method"] == "FINE")]
if len(row_05):
    r = row_05.iloc[0]
    summary_bits.append(
        f"- **Aggregation @ 0.5 d FINE:** slide={r['slide_mean_neg_log']:.3f}, "
        f"pooled={r['pooled_mean_neg_log']:.3f}, Δ={r['pooled_minus_slide']:.3f}; "
        f"events/seed min–max={int(r['events_per_seed_min'])}–{int(r['events_per_seed_max'])}."
    )

miss_05 = missing.loc[np.isclose(missing["horizon_days"], 0.5) & (missing["method"] == "FINE")]
if len(miss_05):
    m = miss_05.iloc[0]
    summary_bits.append(
        f"- **Missing FINE seeds @ 0.5 d:** {m['missing_seeds']} "
        f"(present {m['present_n']}/{m['advertised_n']}); empty npz={m['empty_npz_seeds']}."
    )

tail_05 = tails.loc[np.isclose(tails["horizon_days"], 0.5)]
if len(tail_05):
    fine_t = tail_05.loc[tail_05["method"] == "FINE"].iloc[0]
    etas_t = tail_05.loc[tail_05["method"] == "etas"].iloc[0]
    summary_bits.append(
        f"- **Heavy tail @ 0.5 d:** FINE frac(M≥5)={fine_t['frac_m_ge_5']:.4f} vs ETAS {etas_t['frac_m_ge_5']:.4f}."
    )

pos_05 = pos_summary.loc[np.isclose(pos_summary["horizon_days"], 0.5) & (pos_summary["method"] == "FINE")]
if len(pos_05):
    p = pos_05.iloc[0]
    summary_bits.append(
        f"- **Position @ 0.5 d FINE:** first-event −log={p['first_mean_neg_log']:.3f}, "
        f"later={p['later_mean_neg_log']:.3f}; first frac(M≥5)={p['first_frac_m_ge_5']:.4f}."
    )

same_05 = same_m.loc[np.isclose(same_m["horizon_days"], 0.5)]
if len(same_05):
    s = same_05.iloc[0]
    summary_bits.append(
        f"- **Same-m @ 0.5 d:** MAGNET wins on {100*s['frac_magnet_wins']:.1f}% of FINE draws; "
        f"mean NLL MAGNET={s['mean_nll_magnet']:.3f} vs GR={s['mean_nll_gr_on_same_m']:.3f}."
    )

summary_bits.append(
    "\n**Primary cause (Phase A):** first-in-window FINE draws are heavy/bad at short H "
    "(0.5 d first −log≈1.33, frac(M≥5)≈9%; later −log≈−0.07). Mix of (i) heavy first draws + (ii) skewed/empty seeds; "
    "not (iii) later-window drift; (iv) aggregation only explains slide vs pooled, not FINE vs ETAS."
)
summary_bits.append(
    "\n**Next:** Phase B probes in the guide — observed next-event vs first-draw generative."
)

display(Markdown("\n".join(["### Auto snapshot", ""] + summary_bits)))
